## This notebook is built to separate out the dual tcr cells from the single tcr cells

In [14]:
import polars as pl
import numpy as np
import pandas as pd

In [15]:
b10br_df = pl.read_csv('../Data/20260116 Comparison 3/20251223 BL6-B10BR HTx HIL Clonotypes with ADT Counts.csv')
balbc_df = pl.read_csv('../Data/20260116 Comparison 3/20251218 BL6-BALBc HTx HIL Clonotypes with ADT Counts.csv')
d7abc_df = pl.read_csv('../Data/20260116 Comparison 3/20260114 B10BR D7ABC Kb LL Repertoire + ADT counts.csv')
b10br_dual_df = pl.read_csv('../Data/20260116 Comparison 3/20260714 BL6-B10BR HTx HIL dual cells.csv')
balbc_dual_df = pl.read_csv('../Data/20260116 Comparison 3/20260722 BL6-BALBc HTx HIL dual cells.csv')

In [16]:
b10br_duals = pl.col('Cell_Index').is_in(b10br_dual_df['cell_id'].to_list())
b10br_single_data = b10br_df.filter(~b10br_duals)
b10br_dual_data = b10br_df.filter(b10br_duals)

balbc_duals = pl.col('Cell_Index').is_in(balbc_dual_df['cell_id'].to_list())
balbc_single_data = balbc_df.filter(~balbc_duals)
balbc_dual_data = balbc_df.filter(balbc_duals)

In [17]:
b10br_dual_data = b10br_dual_data.join(
    b10br_dual_df.select('cell_id', 'TCRclonotype_new'),
    left_on='Cell_Index',
    right_on='cell_id',
    how='left'
)

b10br_cols = b10br_dual_data.columns
b10br_cols.remove('TCRclonotype_new')
insertion = b10br_cols.index('TCRClonotype') + 1
b10br_cols = b10br_cols[:insertion] + ['TCRclonotype_new'] + b10br_cols[insertion:]
b10br_dual_data = b10br_dual_data.select(b10br_cols)

balbc_dual_data = balbc_dual_data.join(
    balbc_dual_df.select('cell_id', 'TCRclonotype_new'),
    left_on='Cell_Index',
    right_on='cell_id',
    how='left'
)

balbc_cols = balbc_dual_data.columns
balbc_cols.remove('TCRclonotype_new')
insertion = balbc_cols.index('TCRClonotype') + 1
balbc_cols = balbc_cols[:insertion] + ['TCRclonotype_new'] + balbc_cols[insertion:]
balbc_dual_data = balbc_dual_data.select(balbc_cols)

In [18]:
b10br_single_data.write_csv('Comparison3_Samplewise_Outputs/B10BR_HIL/B10BR Single Cells.csv')
b10br_dual_data.write_csv('Comparison3_Samplewise_Outputs/B10BR_HIL/B10BR Dual Cells.csv')

balbc_single_data.write_csv('Comparison3_Samplewise_Outputs/BALBC_HIL/BALBc Single Cells.csv')
balbc_dual_data.write_csv('Comparison3_Samplewise_Outputs/BALBC_HIL/BALBc Dual Cells.csv')

## Building coherence summaries

Now building coherence summaries for the dual and single separated data woof

In [26]:
def filter_zero_dextramer_cells(df, markers, verbose=True):
    """Remove cells where the sum of all dextramer marker counts is zero"""
    dex_sum = df[markers].sum(axis=1)
    mask = dex_sum > 0
    n_before = len(df)
    n_after = int(mask.sum())
    n_removed = n_before - n_after
    
    if verbose:
        pct_removed = 100.0 * n_removed / n_before if n_before > 0 else 0.0
        print(f"[filter_zero_dextramer] Removed {n_removed:,} / {n_before:,} cells "
            f"({pct_removed:.2f}%) with zero total dextramer counts")
    
    return df[mask].copy()

In [27]:
def row_normalise(
    X, eps=0.0, debug=False, tag="P"):
    """Converts raw counts to a frequency distribution, returns zero for zero-dex
    counts"""
    X = np.asarray(X, dtype=float)
    rs = X.sum(axis=1, keepdims=True)

    zero = (rs.squeeze() <= 0)
    if debug:
        frac = 100.0 * float(np.mean(zero)) if X.shape[0] else 0.0
        print(f"[DEBUG] {tag}: {frac:.2f}% cells have zero RiO mass (all peptides 0).")

    rs_safe = rs.copy()
    rs_safe[rs_safe <= 0] = 1.0
    P = X / rs_safe

    if eps > 0:
        P = np.maximum(P, eps)
        P = P / P.sum(axis=1, keepdims=True)

    return P

In [28]:
def mean_pairwise_cosine_similarity(P):
    """Returns mean pairwise cosine similarity across non-zero rows"""
    nonzero_mask = P.sum(axis=1) > 0
    P = P[nonzero_mask]
    n = P.shape[0]
    if n < 2:
        return np.nan
    norms = np.linalg.norm(P, axis=1, keepdims=True)
    U = P / norms
    sum_vector = U.sum(axis=0)
    sq_norm_sum = float(sum_vector @ sum_vector)
    return (sq_norm_sum - n) / (n * (n-1))

In [29]:
def clonotype_summary(df, ct_col, markers, tcrbeta_col, min_size=5, sample_col=None):
    '''Coherence, mean TCRbeta (raw and log), mean total dextramer per clonotype
    (raw and log), and sample if sample_col is passed'''
    group_cols = [sample_col, ct_col] if sample_col else [ct_col]
    rows = []

    for key, g in df.groupby(group_cols):
        if len(g) < min_size:
            continue
        X = g[markers].to_numpy(dtype=float)
        P = row_normalise(X)

        total_dex = X.sum(axis=1)
        tcr = g[tcrbeta_col].to_numpy(dtype=float)

        row = {
            'clonotype': key[1] if sample_col else key[0],
            'coherence': mean_pairwise_cosine_similarity(P),
            'mean_tcrbeta': tcr.mean(),
            'mean_total_dextramer': total_dex.mean(),
            'mean_log_tcrbeta': np.log10(tcr+1).mean(),
            'mean_log_total_dextramer': np.log10(total_dex).mean(),
            'n_cells': len(g)
            }

        if sample_col:
            row['sample'] = key[0]
        rows.append(row)
    df_out = pd.DataFrame(rows)
    cols = (['sample', 'clonotype'] if sample_col else ['clonotype']) + \
        ['coherence', 'mean_tcrbeta', 'mean_total_dextramer',
        'mean_log_tcrbeta', 'mean_log_total_dextramer', 'n_cells']

    return df_out[cols]

In [36]:
markers_b10br = [
    "RiO-Allo:H-2Kb-ATLVFHNL-pAbO",
    "RiO-Allo:H-2Kb-EEEPVKKI-pAbO",
    "RiO-Allo:H-2Kb-HIYEFPQL-pAbO",
    "RiO-Allo:H-2Kb-INFDFPKL-pAbO",
    "RiO-Allo:H-2Kb-RAYLFNSV-pAbO",
    "RiO-Allo:H-2Kb-RTYTYEKL-pAbO",
    "RiO-Allo:H-2Kb-SNYLFTKL-pAbO",
    "RiO-Allo:H-2Kb-SSYTFPKM-pAbO",
    "RiO-Allo:H-2Kb-SVYVYKVL-pAbO",
    "RiO-Allo:H-2Kb-VAFDFTKV-pAbO",
    "RiO-Allo:H-2Kb-VGPRYTNL-pAbO",
    "RiO-Allo:H-2Kb-VIVRFLTV-pAbO",
    "RiO-Allo:H-2Kb-VSFTYRYL-pAbO",
]

markers_balbc = [
    "RiO-Allo:H-2Kb-ATLVFHNL-pAbO",
    "RiO-Allo:H-2Kb-HIYEFPQL-pAbO",
    "RiO-Allo:H-2Kb-INFDFPKL-pAbO",
    "RiO-Allo:H-2Kb-RAYLFNSV-pAbO",
    "RiO-Allo:H-2Kb-RTYTYEKL-pAbO",
    "RiO-Allo:H-2Kb-SNYLFTKL-pAbO",
    "RiO-Allo:H-2Kb-SSYTFPKM-pAbO",
    "RiO-Allo:H-2Kb-SVYVYKVL-pAbO",
    "RiO-Allo:H-2Kb-VAFDFTKV-pAbO",
    "RiO-Allo:H-2Kb-VGPRYTNL-pAbO",
    "RiO-Allo:H-2Kb-VIVRFLTV-pAbO",
    "RiO-Allo:H-2Kb-VSFTYRYL-pAbO",
    "RiO-H-2:H-2Kd-SYFPEITHI-ADEX5099-pAbO"
]

tcrbeta_col = 'TCR-beta-Tcrb-AMM2021-pAbO'
CT_COL = "TCRClonotype"
SAMPLE_COL = "Sample_Origin"
min_size=5

In [37]:
b10br_single_pd = filter_zero_dextramer_cells(
    pd.DataFrame(b10br_single_data.to_dict(as_series=False)), markers_b10br
    )

b10br_dual_pd = filter_zero_dextramer_cells(
    pd.DataFrame(b10br_dual_data.to_dict(as_series=False)), markers_b10br
    )

balbc_single_pd = filter_zero_dextramer_cells(
    pd.DataFrame(balbc_single_data.to_dict(as_series=False)), markers_balbc
    )

balbc_dual_pd = filter_zero_dextramer_cells(
    pd.DataFrame(balbc_dual_data.to_dict(as_series=False)), markers_balbc
    )

[filter_zero_dextramer] Removed 110 / 17,681 cells (0.62%) with zero total dextramer counts
[filter_zero_dextramer] Removed 0 / 1,247 cells (0.00%) with zero total dextramer counts
[filter_zero_dextramer] Removed 112 / 8,649 cells (1.29%) with zero total dextramer counts
[filter_zero_dextramer] Removed 0 / 670 cells (0.00%) with zero total dextramer counts


In [42]:
coh_b10br_single_sw = clonotype_summary(
    b10br_single_pd, CT_COL, markers_b10br, tcrbeta_col,
    min_size=min_size, sample_col=SAMPLE_COL
)

coh_b10br_single_agg = clonotype_summary(
    b10br_single_pd, CT_COL, markers_b10br, tcrbeta_col,
    min_size=min_size
)

coh_b10br_dual_sw = clonotype_summary(
    b10br_dual_pd, "TCRclonotype_new", markers_b10br, tcrbeta_col,
    min_size=min_size, sample_col=SAMPLE_COL
)

coh_b10br_dual_agg = clonotype_summary(
    b10br_dual_pd, "TCRclonotype_new", markers_b10br, tcrbeta_col,
    min_size=min_size
)

In [43]:
coh_balbc_single_sw = clonotype_summary(
    balbc_single_pd, CT_COL, markers_balbc, tcrbeta_col,
    min_size=min_size, sample_col=SAMPLE_COL
)

coh_balbc_single_agg = clonotype_summary(
    balbc_single_pd, CT_COL, markers_balbc, tcrbeta_col,
    min_size=min_size
)

coh_balbc_dual_sw = clonotype_summary(
    balbc_dual_pd, "TCRclonotype_new", markers_balbc, tcrbeta_col,
    min_size=min_size, sample_col=SAMPLE_COL
)

coh_balbc_dual_agg = clonotype_summary(
    balbc_dual_pd, "TCRclonotype_new", markers_balbc, tcrbeta_col,
    min_size=min_size
)

In [44]:
coh_b10br_single_sw.to_csv('Comparison3_Samplewise_Outputs/B10BR_HIL/b10br_clonotype_summary_single_zerodex_sw.csv', index=False)

coh_b10br_single_agg.to_csv('Comparison3_Samplewise_Outputs/B10BR_HIL/b10br_clonotype_summary_single_zerodex_agg.csv', index=False)

coh_b10br_dual_sw.to_csv('Comparison3_Samplewise_Outputs/B10BR_HIL/b10br_clonotype_summary_dual_zerodex_sw.csv', index=False)

coh_b10br_dual_agg.to_csv('Comparison3_Samplewise_Outputs/B10BR_HIL/b10br_clonotype_summary_dual_zerodex_agg.csv', index=False)

In [45]:
coh_balbc_single_sw.to_csv('Comparison3_Samplewise_Outputs/BALBc_HIL/balbc_clonotype_summary_single_zerodex_sw.csv', index=False)

coh_balbc_single_agg.to_csv('Comparison3_Samplewise_Outputs/BALBc_HIL/balbc_clonotype_summary_single_zerodex_agg.csv', index=False)

coh_balbc_dual_sw.to_csv('Comparison3_Samplewise_Outputs/BALBc_HIL/balbc_clonotype_summary_dual_zerodex_sw.csv', index=False)

coh_balbc_dual_agg.to_csv('Comparison3_Samplewise_Outputs/BALBc_HIL/balbc_clonotype_summary_dual_zerodex_agg.csv', index=False)